<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-geollms/blob/main/phase1_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NDVI dataset preprocessing

In [ ]:
# Import required libraries
import os                              # For interacting with the operating system (e.g., file paths, checking folder existence)
import pandas as pd                   # For data manipulation and analysis using DataFrames
import requests                       # For making HTTP requests (used later if needed to fetch data from the web)
from io import StringIO               # To read string data as file-like objects (useful for requests)
import networkx as nx                 # For working with graph structures (used in later steps)
from scipy.spatial import cKDTree     # For fast spatial indexing and nearest-neighbor lookup
import folium                         # For creating interactive maps
from math import sqrt                 # For mathematical operations, including distance calculations
from geopy.geocoders import Nominatim # For converting addresses into geographic coordinates (geocoding)

# Step 1: Clone the GitHub repository if not already present
repo_url = "https://github.com/Dr-Isam-ALJAWARNEH/fds-project-geollms.git"  # URL of the GitHub repository
clone_dir = "fds-project-geollms"                                           # Local folder name where the repo will be cloned

# Check if the repository is already cloned to avoid re-downloading
if not os.path.exists(clone_dir):                                           # If the folder does not exist
    os.system(f"git clone {repo_url}")                                      # Clone the repository using a terminal command

# Step 2: Path to NDVI folder
ndvi_folder = os.path.join(clone_dir, "Datasets", "NDVI")                  # Construct the full path to the NDVI dataset folder inside the repo

# Step 3: Read and analyze each CSV
all_dfs = []                                                                # Initialize an empty list to store DataFrames for each NDVI CSV file
print("Reading files from NDVI folder...\n")                               # Print message indicating reading process is starting

# Loop through all files in the NDVI folder
for file in os.listdir(ndvi_folder):                                       # Iterate over all files in the folder
    if file.endswith(".csv"):                                              # Only consider files with a .csv extension
        file_path = os.path.join(ndvi_folder, file)                        # Get the full path to the current CSV file
        df = pd.read_csv(file_path)                                        # Read the CSV into a pandas DataFrame

        # Check and report missing values in each file
        missing = df.isnull().sum()                                        # Count missing (NaN) values for each column
        print(f" File: {file}")                                            # Print file name
        print(f" → Rows: {len(df)}, Columns: {len(df.columns)}")           # Print the number of rows and columns
        print(f" → Missing values:\n{missing}\n")                          # Print the count of missing values per column

        all_dfs.append(df)                                                 # Append the DataFrame to the list of all NDVI DataFrames

# Step 4: Combine all into one DataFrame
combined_ndvi_df = pd.concat(all_dfs, ignore_index=True)                  # Concatenate all individual DataFrames into one, resetting the index

# Step 5: Final summary
print(" Combined NDVI Dataset Info:")                                      # Print message for the final summary
print(combined_ndvi_df.info())                                             # Print summary info of the combined DataFrame (columns, types, non-null counts)


AQ Dataset Preprocessing

In [ ]:
# URL for GitHub files
# This is the base URL pointing to the folder containing all the AQ (Air Quality) CSV datasets on GitHub
base_url = "https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/AQ_data/"

# List of files to read
# Creating a list of filenames following the naming pattern 'chicago_eclipse_data_part_1.csv' to 'chicago_eclipse_data_part_19.csv'
filenames = [f"chicago_eclipse_data_part_{i}.csv" for i in range(1, 20)]

# List to store DataFrames
# Initializing an empty list to store the individual DataFrames read from each CSV file
dfs = []

# Download and read each CSV into a DataFrame
# Looping through each filename in the list to download and process the data
for filename in filenames:
    # Construct the full URL by appending the filename to the base URL
    url = base_url + filename
    # Send a GET request to download the file content from the URL
    response = requests.get(url)
    # Check if the request was successful (status code 200 means OK)
    if response.status_code == 200:
        # Read the CSV content from the response text into a DataFrame using StringIO to simulate a file-like object
        df = pd.read_csv(StringIO(response.text))
        # Append the DataFrame to the list of DataFrames
        dfs.append(df)
    else:
        # Print an error message if the file failed to load (e.g., if it doesn't exist or there was a connection issue)
        print(f"Failed to load: {filename}")

# Combine all CSV files into one DataFrame
# Concatenate all the individual DataFrames in the list into one large DataFrame, resetting the index
aq_df = pd.concat(dfs, ignore_index=True)

# Check for missing values
# Count how many missing/null values exist in the 'PM25' column (which stores air quality particulate data)
missing_pm25 = aq_df['PM25'].isnull().sum()
# Count missing values in the 'ReadingDateTimeUTC' column (timestamp of the reading)
missing_datetime = aq_df['ReadingDateTimeUTC'].isnull().sum()
# Count missing values in the 'Latitude' column (spatial location of the sensor)
missing_lat = aq_df['Latitude'].isnull().sum()
# Count missing values in the 'Longitude' column (spatial location of the sensor)
missing_lon = aq_df['Longitude'].isnull().sum()

# Print summary of missing values for key columns to understand data quality
print(f"Missing values in 'PM25': {missing_pm25}")
print(f"Missing values in 'ReadingDateTimeUTC': {missing_datetime}")
print(f"Missing values in 'Latitude': {missing_lat}")
print(f"Missing values in 'Longitude': {missing_lon}")


Main Code

In [ ]:
# Initialize geocoder
geolocator = Nominatim(user_agent="geo_path_finder")  # Create a geolocator object to convert location names into geographic coordinates using the OpenStreetMap service

# Function to geocode a place (convert location name to coordinates)
def geocode_place(place_name):
    location = geolocator.geocode(place_name + ", Chicago, IL")  # Append "Chicago, IL" to limit the search to that region
    if location:
        return (location.longitude, location.latitude)  # Return coordinates in (longitude, latitude) format
    else:
        raise ValueError(f"Location '{place_name}' could not be found.")  # Raise error if geocoding fails

# Step 1: Load datasets from GitHub
bike_url = 'https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/OSM%20datasets/chicago_bike_edges_useful_2.csv'
bike_df = pd.read_csv(bike_url, skipinitialspace=True)  # Read the bike network CSV into a DataFrame

# Step 2: Build the bike network graph
G = nx.Graph()  # Initialize an empty undirected graph
for _, row in bike_df.iterrows():  # Iterate over each row of the bike DataFrame
    u = (row['start_lon'], row['start_lat'])  # Define start node as a tuple (longitude, latitude)
    v = (row['end_lon'], row['end_lat'])  # Define end node
    weight = row['length']  # Use the 'length' column as the weight for the edge
    G.add_edge(u, v, length=weight)  # Add edge to the graph with the specified weight

# Step 3: Filter AQ data to use only the latest reading per sensor location
aq_df['ReadingDateTimeUTC'] = pd.to_datetime(aq_df['ReadingDateTimeUTC'])  # Convert date strings to datetime format
latest_aq_df = aq_df.sort_values('ReadingDateTimeUTC').groupby(['Latitude', 'Longitude']).tail(1)  # Keep the most recent record for each (lat, lon)
latest_aq_coords = latest_aq_df[['Latitude', 'Longitude']].values  # Extract coordinates as NumPy array
aq_tree = cKDTree(latest_aq_coords)  # Build a spatial index for AQ data using KDTree

# Prepare latest NDVI points
# If your NDVI data has a time column like 'Date', uncomment next line
# combined_ndvi_df['Date'] = pd.to_datetime(combined_ndvi_df['Date'])
latest_ndvi_df = combined_ndvi_df.sort_values('Date').groupby(['x', 'y']).tail(1)  # Keep the latest NDVI value per (x, y) coordinate

ndvi_coords = latest_ndvi_df[['x', 'y']].values  # Extract NDVI coordinates
ndvi_tree = cKDTree(ndvi_coords)  # Build a KDTree index for NDVI data

# Step 4: Assign PM2.5 and NDVI values to each edge and compute cost
alpha = 0.1  # Weight factor for air quality penalty
beta = 0.05  # Weight factor for NDVI reward

for u, v, data in G.edges(data=True):  # Loop through each edge with its data
    midpoint = ((u[0] + v[0]) / 2, (u[1] + v[1]) / 2)  # Compute the midpoint of the edge
    # Air quality
    dist, idx = aq_tree.query([midpoint[1], midpoint[0]])  # Query closest AQ point (input as lat, lon)
    pm25 = latest_aq_df.iloc[idx]['PM25']  # Get PM2.5 value from AQ data

    # NDVI
    ndvi_dist, ndvi_idx = ndvi_tree.query([midpoint[0], midpoint[1]])  # Query closest NDVI point (x=lon, y=lat)
    ndvi_value = latest_ndvi_df.iloc[ndvi_idx]['grid_code']  # Retrieve NDVI value from 'grid_code'

    # Assign values to edge
    data['pm25'] = pm25
    data['ndvi'] = ndvi_value
    data['base_cost'] = data['weight'] * (1 + alpha * pm25 - beta * ndvi_value)  # Compute weighted cost
    data['cost'] = data['base_cost']  # Assign cost to edge

# Step 5: Get start and end locations
start_name = input("Enter the start location (e.g., Melrose Park): ")  # Ask user for start location
end_name = input("Enter the end location (e.g., Hyde Park): ")  # Ask user for destination

# Convert to coordinates using geopy
try:
    start_point = geocode_place(start_name)  # Get coordinates for start location
    end_point = geocode_place(end_name)  # Get coordinates for end location
    print(f"Start coordinates: {start_point}")
    print(f"End coordinates: {end_point}")
except ValueError as e:
    print(e)  # Print error if geocoding fails
    exit()  # Stop program execution

# Step 6: Find nearest nodes
nodes_list = list(G.nodes())  # Get list of all nodes in the graph
bike_tree = cKDTree([(lat, lon) for lon, lat in nodes_list])  # Build KDTree using (lat, lon) tuples

_, start_idx = bike_tree.query((start_point[1], start_point[0]))  # Find nearest graph node to start_point
_, end_idx = bike_tree.query((end_point[1], end_point[0]))  # Find nearest node to end_point
start_node = nodes_list[start_idx]  # Extract nearest start node
end_node = nodes_list[end_idx]  # Extract nearest end node

# Step 7: Add virtual edges
def euclidean_distance(coord1, coord2):
    return sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)  # Compute Euclidean distance between 2 coordinates

G.add_edge(start_point, start_node, weight=euclidean_distance(start_point, start_node), pm25=0, ndvi=0, base_cost=0, cost=0)  # Add virtual edge from start_point to nearest node
G.add_edge(end_point, end_node, weight=euclidean_distance(end_point, end_node), pm25=0, ndvi=0, base_cost=0, cost=0)  # Add virtual edge from end_point to nearest node

# Step 8: Yen's K-shortest paths with penalties
def yen_k_shortest_paths(graph, source, target, k, penalty_factor=2.0):
    paths = []  # Store list of shortest paths
    for _ in range(k):
        path = nx.shortest_path(graph, source, target, weight='cost')  # Get the current shortest path
        paths.append(path)
        # Penalize reused edges
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            graph[u][v]['cost'] *= penalty_factor  # Increase cost of reused edge
    return paths

top_k_paths = yen_k_shortest_paths(G, start_point, end_point, 5)  # Find 5 best paths

# Step 9: Compute distances, PM2.5, and NDVI for each path
path_info = []  # Store details of each path
for path in top_k_paths:
    total_length = 0
    total_pm25 = 0
    total_ndvi = 0
    for i in range(len(path) - 1):
        edge_data = G.get_edge_data(path[i], path[i + 1])  # Get edge information
        total_length += edge_data['weight']
        total_pm25 += edge_data['pm25']
        total_ndvi += edge_data['ndvi']
    avg_pm25 = total_pm25 / (len(path) - 1)  # Compute average PM2.5
    avg_ndvi = total_ndvi / (len(path) - 1)  # Compute average NDVI
    path_info.append({'path': path, 'distance': total_length / 1000, 'pm25': avg_pm25, 'ndvi': avg_ndvi})  # Store info in km

# Step 10: Ask user for ranking criteria
print("\nChoose the ranking criteria:")
print("1. Shortest path")
print("2. Least polluted path (lowest PM2.5)")
print("3. Greenest path (highest NDVI)")

criteria_choice = input("\nEnter the number corresponding to your choice (1, 2, or 3): ")  # Take user input

# Step 11: Sort paths based on the chosen criterion
if criteria_choice == "1":
    path_info.sort(key=lambda x: x['distance'])  # Sort by distance
    criteria = "Shortest Path"
    print(f"\n Top path distance: {path_info[0]['distance']:.2f} km")  # Print distance of shortest path

elif criteria_choice == "2":
    path_info.sort(key=lambda x: x['pm25'])  # Sort by air pollution (ascending)
    criteria = "Least Polluted Path"
elif criteria_choice == "3":
    path_info.sort(key=lambda x: -x['ndvi'])  # Sort by greenery (descending)
    criteria = "Greenest Path"
else:
    print("Invalid choice! Exiting.")
    exit()

# Step 12: Print path details
print(f"\nRanking based on: {criteria}")
for idx, info in enumerate([path_info[0]] + path_info[1:]):  # Print all path results
    label = "Best" if idx == 0 else f"Path {idx+1}"
    print(f"{label}: {info['distance']:.2f} km, Avg PM2.5: {info['pm25']:.2f}, Avg NDVI: {info['ndvi']:.2f}")

# Step 13: Visualization
m = folium.Map(location=[(start_point[1] + end_point[1]) / 2, (start_point[0] + end_point[0]) / 2], zoom_start=12)  # Center map between start and end

# Draw network edges
for u, v, data in G.edges(data=True):
    coords = [(lat, lon) for lon, lat in [u, v]]  # Flip to (lat, lon) format
    folium.PolyLine(coords, color='lightgray', weight=1, opacity=0.3).add_to(m)  # Add all network edges to map

# Draw paths
colors = ['#228B22', 'blue', 'orange', 'purple', 'brown']
for idx, info in enumerate(path_info):
    coords = [(lat, lon) for lon, lat in info['path']]  # Convert each path to map format
    folium.PolyLine(coords, color=colors[idx], weight=5, opacity=0.8, popup=f"Path {idx+1}: {info['distance']:.2f} km, PM2.5: {info['pm25']:.2f}, NDVI: {info['ndvi']:.2f}").add_to(m)  # Add colored route

# Add start/end markers
folium.Marker(location=(start_point[1], start_point[0]), popup="Start Location", icon=folium.Icon(color='green')).add_to(m)
folium.Marker(location=(end_point[1], end_point[0]), popup="End Location", icon=folium.Icon(color='red')).add_to(m)

# Display the map
m
